# 2.3 — When the English sounds right but the meaning is wrong

(Requires: `pip install sentencepiece sacrebleu`; `transformers` and `torch` are already installed.)

## Q1 — Where does the meaning go?

```mermaid
flowchart LR
    A["Tamil text"] --> B["Tokenizer: subword pieces + tam_Taml tag"]
    B --> C["Token IDs (integers)"]
    C --> D["Encoder: one context vector per piece"]
    D --> E["Decoder: told to write eng_Latn, generates one piece at a time"]
    E -->|"each new piece is fed back in"| E
    E --> F["English pieces"]
    F --> G["Detokenizer: English text"]
```

In [2]:
import re
import collections
import difflib
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sacrebleu.metrics import CHRF

MODEL = "facebook/nllb-200-distilled-600M"
tok = AutoTokenizer.from_pretrained(MODEL, src_lang="tam_Taml")      # Tamil -> pieces
tok_en = AutoTokenizer.from_pretrained(MODEL, src_lang="eng_Latn")   # English -> pieces (for Q5 back-translation)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL)
model.eval()
ENG = tok.convert_tokens_to_ids("eng_Latn")
TAM = tok.convert_tokens_to_ids("tam_Taml")
chrf = CHRF()

def translate(texts, num_beams=4, max_new_tokens=128):
    batch = tok(texts, return_tensors="pt", padding=True)
    with torch.no_grad():
        out = model.generate(**batch, forced_bos_token_id=ENG,
                             num_beams=num_beams, max_new_tokens=max_new_tokens)
    return tok.batch_decode(out, skip_special_tokens=True)

Loading weights: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 512/512 [00:00<00:00, 35013.51it/s]


In [3]:
src = "நான் நேற்று சென்னையில் 42 ரூபாய்க்கு மீன் வாங்கினேன்."
print("SOURCE:", src)

enc = tok(src, return_tensors="pt")
print("\n1. TOKENS:", tok.convert_ids_to_tokens(enc.input_ids[0]))
print("2. IDS:   ", enc.input_ids[0].tolist())

with torch.no_grad():
    enc_out = model.get_encoder()(**enc)
print("3. ENCODER OUTPUT shape:", tuple(enc_out.last_hidden_state.shape),
      "-> (sentences, pieces, numbers per piece)")

with torch.no_grad():
    gen = model.generate(**enc, forced_bos_token_id=ENG, num_beams=1, max_new_tokens=40,
                         output_scores=True, return_dict_in_generate=True)
print("\n4. DECODER, one piece at a time (with the model's confidence):")
for step, logits in enumerate(gen.scores):
    tid = gen.sequences[0, step + 1].item()
    p = torch.softmax(logits[0], dim=-1)[tid].item()
    print(f"   step {step:2d}: {tok.convert_ids_to_tokens(tid):<14} p={p:.2f}")

print("\n5. OUTPUT:", tok.decode(gen.sequences[0], skip_special_tokens=True))

SOURCE: நான் நேற்று சென்னையில் 42 ரூபாய்க்கு மீன் வாங்கினேன்.

1. TOKENS: ['tam_Taml', '▁நான்', '▁நேற்று', '▁செ', 'ன்ன', 'ையில்', '▁42', '▁ரூ', 'பா', 'ய', '்க்க', 'ு', '▁மீ', 'ன்', '▁வாங்க', 'ினேன்', '.', '</s>']
2. IDS:    [256170, 12477, 223057, 5479, 4405, 16608, 12483, 148281, 30159, 248629, 54858, 248289, 25758, 736, 101403, 228160, 248075, 2]


[transformers] Both `max_new_tokens` (=40) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


3. ENCODER OUTPUT shape: (1, 18, 1024) -> (sentences, pieces, numbers per piece)

4. DECODER, one piece at a time (with the model's confidence):
   step  0: eng_Latn       p=1.00
   step  1: ▁I             p=0.73
   step  2: ▁bought        p=0.75
   step  3: ▁fish          p=0.59
   step  4: ▁for           p=0.47
   step  5: ▁Rs            p=0.56
   step  6: ▁42            p=0.66
   step  7: ▁yesterday     p=0.54
   step  8: ▁in            p=0.75
   step  9: ▁Ch            p=0.79
   step 10: enna           p=0.94
   step 11: i              p=0.95
   step 12: .              p=0.79
   step 13: </s>           p=0.90

5. OUTPUT: I bought fish for Rs 42 yesterday in Chennai.


## Q2 — Testing the model under different linguistic situations

In [4]:
TESTS = [
    # baseline
    ("baseline", "இன்று வானிலை மிகவும் வெப்பமாக உள்ளது.", "The weather is very hot today."),
    ("baseline", "மாணவர்கள் நூலகத்தில் புத்தகங்களைப் படிக்கிறார்கள்.", "The students are reading books in the library."),
    ("baseline", "கல்லூரி முடிந்ததும் அவனுக்கு வேலை கிடைத்தது.", "He got a job after finishing college."),
    # names
    ("names", "அருண் தன் தங்கை கீர்த்தனாவுடன் மதுரைக்குச் சென்றான்.", "Arun went to Madurai with his younger sister Keerthana."),
    ("names", "டாக்டர் சுப்பிரமணியம் திருநெல்வேலியில் வசிக்கிறார்.", "Dr. Subramaniam lives in Tirunelveli."),
    ("names", "திருவள்ளுவர் திருக்குறளை எழுதினார்.", "Thiruvalluvar wrote the Thirukkural."),
    # numbers
    ("numbers", "இந்தப் பாலம் 1950-ஆம் ஆண்டு கட்டப்பட்டது.", "This bridge was built in 1950."),
    ("numbers", "ரயில் காலை 7:45 மணிக்குப் புறப்படும்.", "The train will leave at 7:45 in the morning."),
    ("numbers", "அவருடைய ஆண்டுச் சம்பளம் 3.5 லட்சம் ரூபாய்.", "His annual salary is 3.5 lakh rupees."),
    ("numbers", "இரண்டாயிரத்து இருபத்து நான்கு பேர் தேர்வு எழுதினர்.", "Two thousand and twenty-four people took the exam."),
    # context and ambiguity
    ("context", "ஆறு மணிக்கு நாங்கள் ஆற்றில் குளித்தோம்.", "We bathed in the river at six o'clock."),
    ("context", "அவள் நேற்று வந்தாள்.", "She came yesterday."),
    ("context", "அவர் நாளை வருவார்.", "He will come tomorrow."),
    # culturally specific expressions
    ("culture", "அவன் மேலதிகாரியிடம் காக்கா பிடிக்கிறான்.", "He is buttering up his boss."),
    ("culture", "எனக்குத் தலைக்கு மேல் வேலை இருக்கிறது.", "I am swamped with work."),
    ("culture", "பொங்கல் அன்று எல்லோரும் புதுப் பானையில் பொங்கல் வைத்தார்கள்.", "On Pongal, everyone cooked pongal in new pots."),
    ("culture", "அவன் ஒரு கிணற்றுத் தவளை.", "He is narrow-minded."),
    ("culture", "அண்ணா, இந்த பஸ் எங்கே போகும்?", "Brother, where does this bus go?"),
    # spoken Tamil
    ("spoken", "நான் நேத்து கடைக்குப் போனேன்.", "I went to the shop yesterday."),
    ("spoken", "சாப்டியா?", "Did you eat?"),
    ("spoken", "என்ன பண்ணிட்டு இருக்க?", "What are you doing?"),
    ("spoken", "அவன் இன்னும் வரல.", "He hasn't come yet."),
    # code-mixed Tamil-English
    ("tanglish", "நான் tomorrow office போறேன்.", "I am going to the office tomorrow."),
    ("tanglish", "meeting cancel ஆயிடுச்சு.", "The meeting got cancelled."),
    ("tanglish", "இந்த movie செம mass-ஆ இருந்துச்சு.", "This movie was really awesome."),
    ("tanglish", "நாளைக்கு exam இருக்கு, அதனால் படிக்கணும்.", "I have an exam tomorrow, so I need to study."),
]
print(len(TESTS), "sentences")

26 sentences


In [5]:
outputs = translate([src for _, src, _ in TESTS])
scores = [chrf.sentence_score(h, [r]).score for (_, _, r), h in zip(TESTS, outputs)]

for i, ((cat, src, ref), hyp) in enumerate(zip(TESTS, outputs)):
    print(f"[{i:2d}] {cat:<9} chrF {scores[i]:5.1f}")
    print(f"     TA : {src}")
    print(f"     REF: {ref}")
    print(f"     OUT: {hyp}\n")

[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[ 0] baseline  chrF  77.7
     TA : இன்று வானிலை மிகவும் வெப்பமாக உள்ளது.
     REF: The weather is very hot today.
     OUT: Today the weather is very hot.

[ 1] baseline  chrF  90.5
     TA : மாணவர்கள் நூலகத்தில் புத்தகங்களைப் படிக்கிறார்கள்.
     REF: The students are reading books in the library.
     OUT: Students are reading books in the library.

[ 2] baseline  chrF  47.7
     TA : கல்லூரி முடிந்ததும் அவனுக்கு வேலை கிடைத்தது.
     REF: He got a job after finishing college.
     OUT: When he finished college, he got a job.

[ 3] names     chrF  73.5
     TA : அருண் தன் தங்கை கீர்த்தனாவுடன் மதுரைக்குச் சென்றான்.
     REF: Arun went to Madurai with his younger sister Keerthana.
     OUT: Arun went to Madurai with his sister Keertana.

[ 4] names     chrF  70.3
     TA : டாக்டர் சுப்பிரமணியம் திருநெல்வேலியில் வசிக்கிறார்.
     REF: Dr. Subramaniam lives in Tirunelveli.
     OUT: Dr. Subramanian lives in Thranelveli.

[ 5] names     chrF  51.9
     TA : திருவள்ளுவர் திருக்குறளை எழுதின

In [6]:
# my verdicts after reading every output: ok / minor / fluent-wrong / broken
VERDICT = {
    0: "ok", 1: "ok", 2: "ok",
    3: "minor", 4: "fluent-wrong", 5: "fluent-wrong",
    6: "ok", 7: "ok", 8: "ok", 9: "ok",
    10: "ok", 11: "ok", 12: "minor",
    13: "fluent-wrong", 14: "minor", 15: "minor", 16: "fluent-wrong", 17: "fluent-wrong",
    18: "ok", 19: "fluent-wrong", 20: "ok", 21: "ok",
    22: "ok", 23: "ok", 24: "minor", 25: "ok",
}
LABELS = ["ok", "minor", "fluent-wrong", "broken"]
ORDER = ["baseline", "names", "numbers", "context", "culture", "spoken", "tanglish"]

print(f"{'category':<10}" + "".join(f"{v:>14}" for v in LABELS))
for cat in ORDER:
    idx = [i for i, (c, _, _) in enumerate(TESTS) if c == cat]
    counts = collections.Counter(VERDICT[i] for i in idx)
    print(f"{cat:<10}" + "".join(f"{counts[v]:>14}" for v in LABELS))

print("\nchrF by verdict:")
for v in LABELS:
    s = [scores[i] for i in VERDICT if VERDICT[i] == v]
    if s:
        print(f"  {v:<13} mean {np.mean(s):5.1f}   range {min(s):5.1f} - {max(s):5.1f}   n={len(s)}")

category              ok         minor  fluent-wrong        broken
baseline               3             0             0             0
names                  0             1             2             0
numbers                4             0             0             0
context                2             1             0             0
culture                0             2             3             0
spoken                 3             0             1             0
tanglish               3             1             0             0

chrF by verdict:
  ok            mean  69.9   range  28.1 - 100.0   n=15
  minor         mean  57.5   range  14.1 - 100.0   n=5
  fluent-wrong  mean  40.1   range   7.5 -  71.9   n=6


In [7]:
# Q2(c): small changes to the source
PAIRS = [
    ("negation (meaning changes)", "அவன் வந்தான்.", "அவன் வரவில்லை."),
    ("number (meaning changes)", "நான் 42 ரூபாய் கொடுத்தேன்.", "நான் 24 ரூபாய் கொடுத்தேன்."),
    ("gender (meaning changes)", "அவள் பாடம் படித்தாள்.", "அவன் பாடம் படித்தான்."),
    ("written vs spoken negation (same meaning)", "அவன் வரவில்லை.", "அவன் வரல."),
    ("written vs spoken (same meaning)", "நான் நேற்று கடைக்குச் சென்றேன்.", "நான் நேத்து கடைக்குப் போனேன்."),
    ("written vs spoken 'did you eat' (same meaning)", "சாப்பிட்டாயா?", "சாப்டியா?"),
    ("question mark removed (same meaning)", "சாப்டியா?", "சாப்டியா"),
    ("short form of a name (same meaning)", "திருவள்ளுவர் திருக்குறளை எழுதினார்.", "வள்ளுவர் திருக்குறளை எழுதினார்."),
]

for label, a, b in PAIRS:
    ta, tb = translate([a, b])
    print(f"{label}\n   {a}  ->  {ta}\n   {b}  ->  {tb}\n")

[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


negation (meaning changes)
   அவன் வந்தான்.  ->  He came.
   அவன் வரவில்லை.  ->  He didn't come.



[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


number (meaning changes)
   நான் 42 ரூபாய் கொடுத்தேன்.  ->  I gave 42 rupees.
   நான் 24 ரூபாய் கொடுத்தேன்.  ->  I gave you 24 rupees.



[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

written vs spoken 'did you eat' (same meaning)
   சாப்பிட்டாயா?  ->  Have you eaten?
   சாப்டியா?  ->  What is it, Chaptia?



[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


question mark removed (same meaning)
   சாப்டியா?  ->  What is it, Chaptia?
   சாப்டியா  ->  Shaptia



[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


short form of a name (same meaning)
   திருவள்ளுவர் திருக்குறளை எழுதினார்.  ->  Thiruvananthapuram wrote the Thiruvananthapuram.
   வள்ளுவர் திருக்குறளை எழுதினார்.  ->  The villain wrote the crime.



## Q3 — Fixing a repeated failure: names

In [8]:
# Known names written in English letters inside the Tamil sentence before translation.
# Tamil endings ("-இல்" in, "-க்கு" to, ...) are kept after a hyphen, the way Tanglish is written.
# Names used while developing the fix (from the Q2 test set):
DEV_GLOSSARY = {
    "அருண்": "Arun", "கீர்த்தனா": "Keerthana", "மதுரை": "Madurai",
    "சுப்பிரமணியம்": "Subramaniam", "திருநெல்வேலி": "Tirunelveli",
    "திருவள்ளுவர்": "Thiruvalluvar", "வள்ளுவர்": "Valluvar", "திருக்குறள்": "Thirukkural",
}

VOWEL_SIGN = {"ா": "ஆ", "ி": "இ", "ீ": "ஈ", "ு": "உ", "ூ": "ஊ", "ெ": "எ",
              "ே": "ஏ", "ை": "ஐ", "ொ": "ஒ", "ோ": "ஓ", "ௌ": "ஔ"}

def protect_names(text, glossary):
    for ta, en in sorted(glossary.items(), key=lambda kv: -len(kv[0])):
        stems = [ta[:-1] if ta.endswith("்") else ta]   # e.g. திருக்குறள் also matches திருக்குறளை
        if ta.endswith("ம்"):
            stems.append(ta[:-2] + "த்")                 # e.g. எட்டயபுரம் -> எட்டயபுரத்தில்
        for stem in stems:
            def repl(m, en=en):
                suffix = m.group(1)
                if suffix in ("", "்"):
                    return en
                if suffix[0] in VOWEL_SIGN:                 # ை -> ஐ, so the ending is readable on its own
                    suffix = VOWEL_SIGN[suffix[0]] + suffix[1:]
                return f"{en}-{suffix}"
            text = re.sub(r"(?<!\S)" + re.escape(stem) + r"(\S*?)(?=[\s.,?!]|$)", repl, text)
    return text

def translate_fixed(texts, glossary):
    return translate([protect_names(t, glossary) for t in texts])

dev_srcs = [s for c, s, _ in TESTS if c == "names"] + [PAIRS[-1][2]]   # names sentences + "வள்ளுவர்" variant
for s, before, after in zip(dev_srcs, translate(dev_srcs), translate_fixed(dev_srcs, DEV_GLOSSARY)):
    print(f"TA      : {s}\nPROTECTED: {protect_names(s, DEV_GLOSSARY)}\nBEFORE  : {before}\nAFTER   : {after}\n")

[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


TA      : அருண் தன் தங்கை கீர்த்தனாவுடன் மதுரைக்குச் சென்றான்.
PROTECTED: Arun தன் தங்கை Keerthana-வுடன் Madurai-க்குச் சென்றான்.
BEFORE  : Arun went to Madurai with his sister Keertana.
AFTER   : Arun went to Madurai with his sister Keerthana.

TA      : டாக்டர் சுப்பிரமணியம் திருநெல்வேலியில் வசிக்கிறார்.
PROTECTED: டாக்டர் Subramaniam Tirunelveli-யில் வசிக்கிறார்.
BEFORE  : Dr. Subramanian lives in Thranelveli.
AFTER   : Dr. Subramaniam is from Tirunelveli.

TA      : திருவள்ளுவர் திருக்குறளை எழுதினார்.
PROTECTED: Thiruvalluvar Thirukkural-ஐ எழுதினார்.
BEFORE  : Thiruvananthapuram wrote the Thiruvananthapuram.
AFTER   : Thiruvalluvar wrote Thirukkural.

TA      : வள்ளுவர் திருக்குறளை எழுதினார்.
PROTECTED: Valluvar Thirukkural-ஐ எழுதினார்.
BEFORE  : The villain wrote the crime.
AFTER   : Valluvar wrote Thirukkural.



In [9]:
# Held-out sentences: NOT used while developing the fix.
# In a real archive, the glossary would come from catalogue metadata (speaker names, places),
# so the new names are added to the glossary, but the rules above are not changed.
HELDOUT = [
    ("கண்ணகி மதுரையை எரித்தாள்.", "Kannagi burned Madurai.", ["Kannagi", "Madurai"]),
    ("பாரதியார் எட்டயபுரத்தில் பிறந்தார்.", "Bharathiyar was born in Ettayapuram.", ["Bharathiyar", "Ettayapuram"]),
    ("செல்வி கோயம்புத்தூரில் வேலை செய்கிறாள்.", "Selvi works in Coimbatore.", ["Selvi", "Coimbatore"]),
    ("ராஜராஜ சோழன் தஞ்சாவூர் பெரிய கோயிலைக் கட்டினான்.", "Rajaraja Chola built the Big Temple in Thanjavur.", ["Rajaraja Chola", "Thanjavur"]),
    ("இளங்கோவடிகள் சிலப்பதிகாரத்தை எழுதினார்.", "Ilango Adigal wrote the Silappatikaram.", ["Ilango Adigal", "Silappatikaram"]),
    ("முத்துலட்சுமி ரெட்டி ஒரு மருத்துவர்.", "Muthulakshmi Reddy is a doctor.", ["Muthulakshmi Reddy"]),
    ("வேலு தூத்துக்குடிக்குச் சென்றான்.", "Velu went to Thoothukudi.", ["Velu", "Thoothukudi"]),
    ("நாளை காவ்யாவும் பிரவீனும் கன்னியாகுமரிக்கு வருவார்கள்.", "Kavya and Praveen will come to Kanyakumari tomorrow.", ["Kavya", "Praveen", "Kanyakumari"]),
    ("அருணா நாளை வருவாள்.", "Aruna will come tomorrow.", ["Aruna"]),   # NOT in the glossary, but starts like "Arun"
]
FULL_GLOSSARY = {**DEV_GLOSSARY,
    "கண்ணகி": "Kannagi", "பாரதியார்": "Bharathiyar", "எட்டயபுரம்": "Ettayapuram",
    "செல்வி": "Selvi", "கோயம்புத்தூர்": "Coimbatore", "ராஜராஜ சோழன்": "Rajaraja Chola",
    "தஞ்சாவூர்": "Thanjavur", "இளங்கோவடிகள்": "Ilango Adigal", "சிலப்பதிகாரம்": "Silappatikaram",
    "முத்துலட்சுமி ரெட்டி": "Muthulakshmi Reddy", "வேலு": "Velu", "தூத்துக்குடி": "Thoothukudi",
    "காவ்யா": "Kavya", "பிரவீன்": "Praveen", "கன்னியாகுமரி": "Kanyakumari",
}
for s, _, _ in HELDOUT:
    print(protect_names(s, FULL_GLOSSARY))

Kannagi Madurai-யை எரித்தாள்.
Bharathiyar Ettayapuram-தில் பிறந்தார்.
Selvi Coimbatore-இல் வேலை செய்கிறாள்.
Rajaraja Chola Thanjavur பெரிய கோயிலைக் கட்டினான்.
Ilango Adigal Silappatikaram-தை எழுதினார்.
Muthulakshmi Reddy ஒரு மருத்துவர்.
Velu Thoothukudi-க்குச் சென்றான்.
நாளை Kavya-வும் Praveen-உம் Kanyakumari-க்கு வருவார்கள்.
Arun-ஆ நாளை வருவாள்.


## Q4 — Did the fix actually help?

In [10]:
def name_found(name, text, threshold=0.9):
    # lenient match: allows small spelling differences (e.g. Silappathikaram vs Silappatikaram)
    words = re.findall(r"[A-Za-z]+", text)
    n = len(name.split())
    grams = [" ".join(words[i:i + n]) for i in range(len(words) - n + 1)]
    return any(difflib.SequenceMatcher(None, name.lower(), g.lower()).ratio() >= threshold for g in grams)

h_src = [s for s, _, _ in HELDOUT]
h_before = translate(h_src)
h_after = translate_fixed(h_src, FULL_GLOSSARY)

tot = found_b = found_a = 0
chrf_b, chrf_a = [], []
for i, ((s, ref, names), b, a) in enumerate(zip(HELDOUT, h_before, h_after)):
    nb = sum(name_found(n, b) for n in names)
    na = sum(name_found(n, a) for n in names)
    tot, found_b, found_a = tot + len(names), found_b + nb, found_a + na
    cb, ca = chrf.sentence_score(b, [ref]).score, chrf.sentence_score(a, [ref]).score
    chrf_b.append(cb); chrf_a.append(ca)
    print(f"[{i}] REF   : {ref}")
    print(f"    BEFORE: {b}   (names {nb}/{len(names)}, chrF {cb:.1f})")
    print(f"    AFTER : {a}   (names {na}/{len(names)}, chrF {ca:.1f})\n")

print(f"names preserved: before {found_b}/{tot}, after {found_a}/{tot}")
print(f"mean chrF:       before {np.mean(chrf_b):.1f}, after {np.mean(chrf_a):.1f}")

[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[0] REF   : Kannagi burned Madurai.
    BEFORE: Kanaki burned the Madurai.   (names 1/2, chrF 58.1)
    AFTER : Kannagi burned Madurai.   (names 2/2, chrF 100.0)

[1] REF   : Bharathiyar was born in Ettayapuram.
    BEFORE: Bharathiar was born in Etaipuram.   (names 1/2, chrF 67.7)
    AFTER : Bharathiyar was born in Ettayapuram.   (names 2/2, chrF 100.0)

[2] REF   : Selvi works in Coimbatore.
    BEFORE: She works in Coimbatore.   (names 1/2, chrF 78.4)
    AFTER : Selvi works in Coimbatore.   (names 2/2, chrF 100.0)

[3] REF   : Rajaraja Chola built the Big Temple in Thanjavur.
    BEFORE: Rajaraja Sojan built a great temple in Thanjavur.   (names 1/2, chrF 58.9)
    AFTER : Rajaraja Chola Thanjavur built a great temple.   (names 2/2, chrF 57.6)

[4] REF   : Ilango Adigal wrote the Silappatikaram.
    BEFORE: Young men wrote some poems.   (names 0/2, chrF 12.1)
    AFTER : Ilango Adigal wrote the Silappatikaram.   (names 2/2, chrF 100.0)

[5] REF   : Muthulakshmi Reddy is a doctor.


In [11]:
# my verdicts on the held-out outputs (fill in after reading): ok / minor / fluent-wrong / broken
HELDOUT_VERDICT_BEFORE = {i: None for i in range(len(HELDOUT))}
HELDOUT_VERDICT_AFTER = {i: None for i in range(len(HELDOUT))}

for name, vd in [("before", HELDOUT_VERDICT_BEFORE), ("after", HELDOUT_VERDICT_AFTER)]:
    filled = [v for v in vd.values() if v]
    if filled:
        print(name, dict(collections.Counter(filled)))

In [12]:
# Side effects: does the fix change sentences that have no names in the glossary?
fixed_all = translate_fixed([s for _, s, _ in TESTS], FULL_GLOSSARY)
changed = [(i, outputs[i], fixed_all[i]) for i in range(len(TESTS)) if outputs[i] != fixed_all[i]]
print(f"{len(changed)} of {len(TESTS)} Q2 outputs changed")
for i, b, a in changed:
    print(f"[{i:2d}] {TESTS[i][0]:<9} BEFORE: {b}\n               AFTER : {a}")

[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


3 of 26 Q2 outputs changed
[ 3] names     BEFORE: Arun went to Madurai with his sister Keertana.
               AFTER : Arun went to Madurai with his sister Keerthana.
[ 4] names     BEFORE: Dr. Subramanian lives in Thranelveli.
               AFTER : Dr. Subramaniam is from Tirunelveli.
[ 5] names     BEFORE: Thiruvananthapuram wrote the Thiruvananthapuram.
               AFTER : Thiruvalluvar wrote Thirukkural.


## Q5 — Signs of an unreliable translation without a reference

In [13]:
def signals(src):
    batch = tok([src], return_tensors="pt")
    with torch.no_grad():
        out = model.generate(**batch, forced_bos_token_id=ENG, num_beams=4, num_return_sequences=4,
                             max_new_tokens=128, output_scores=True, return_dict_in_generate=True)
    hyps = tok.batch_decode(out.sequences, skip_special_tokens=True)
    best = hyps[0]

    # 1. confidence: average per-piece probability of the chosen translation
    confidence = float(torch.exp(out.sequences_scores[0]))

    # 2. agreement: how similar the other beam candidates are to the chosen one
    agreement = float(np.mean([chrf.sentence_score(h, [best]).score for h in hyps[1:]]))

    # 3. round trip: translate back to Tamil and compare with the original
    eb = tok_en([best], return_tensors="pt")
    with torch.no_grad():
        back_ids = model.generate(**eb, forced_bos_token_id=TAM, num_beams=4, max_new_tokens=128)
    back = tok_en.batch_decode(back_ids, skip_special_tokens=True)[0]
    roundtrip = chrf.sentence_score(back, [src]).score

    # 4. digits in the source that are missing from the output
    nums = re.findall(r"\d+(?:[.:]\d+)?", src)
    numbers_ok = all(n in best for n in nums)

    return {"best": best, "back": back, "confidence": confidence,
            "agreement": agreement, "roundtrip": roundtrip, "numbers_ok": numbers_ok}

dev_sig = [signals(s) for _, s, _ in TESTS]
print(f"{'#':>3} {'verdict':<13}{'conf':>6}{'agree':>7}{'round':>7}{'nums':>6}  output  |  back-translation")
for i, s in enumerate(dev_sig):
    print(f"{i:>3} {VERDICT[i]:<13}{s['confidence']:>6.2f}{s['agreement']:>7.1f}{s['roundtrip']:>7.1f}"
          f"{str(s['numbers_ok']):>6}  {s['best']}  |  {s['back']}")

[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

  # verdict        conf  agree  round  nums  output  |  back-translation
  0 ok             0.72   65.0   55.4  True  Today the weather is very hot.  |  இன்று வானிலை மிகவும் சூடாக இருக்கிறது.
  1 ok             0.76   76.9  100.0  True  Students are reading books in the library.  |  மாணவர்கள் நூலகத்தில் புத்தகங்களைப் படிக்கிறார்கள்.
  2 ok             0.70   57.6   85.9  True  When he finished college, he got a job.  |  கல்லூரி முடிந்ததும், அவருக்கு வேலை கிடைத்தது.
  3 minor          0.67   88.0   48.8  True  Arun went to Madurai with his sister Keertana.  |  அருண் தனது சகோதரியான கீர்தானாவுடன் மதுரை சென்றார்.
  4 fluent-wrong   0.59   64.2   71.2  True  Dr. Subramanian lives in Thranelveli.  |  டாக்டர் சுப்ரமணியன் த்ரணெல்வேலியில் வசிக்கிறார்.
  5 fluent-wrong   0.58   80.7   45.0  True  Thiruvananthapuram wrote the Thiruvananthapuram.  |  திருவனந்தபுரம் திருவனந்தபுரத்தை எழுதினார்.
  6 ok             0.75   77.3   59.0  True  The bridge was built in 1950.  |  இந்த பாலம் 1950ல் கட்டப்பட்

In [14]:
good = [i for i in VERDICT if VERDICT[i] == "ok"]
bad = [i for i in VERDICT if VERDICT[i] == "fluent-wrong"]

for key in ["confidence", "agreement", "roundtrip"]:
    g = [dev_sig[i][key] for i in good]
    b = [dev_sig[i][key] for i in bad]
    print(f"{key:<11} ok: mean {np.mean(g):6.2f} (range {min(g):.2f}-{max(g):.2f})   "
          f"fluent-wrong: mean {np.mean(b):6.2f} (range {min(b):.2f}-{max(b):.2f})")

confidence  ok: mean   0.72 (range 0.57-0.83)   fluent-wrong: mean   0.54 (range 0.36-0.64)
agreement   ok: mean  69.95 (range 22.08-85.68)   fluent-wrong: mean  64.13 (range 48.50-80.68)
roundtrip   ok: mean  54.58 (range 6.92-100.00)   fluent-wrong: mean  37.97 (range 3.21-71.24)


In [15]:
# Flag rule. Thresholds = 25th percentile of the "ok" sentences on the Q2 set,
# i.e. accept that about a quarter of good translations get flagged, then check what is caught.
TH = {k: np.percentile([dev_sig[i][k] for i in good], 25) for k in ["confidence", "agreement", "roundtrip"]}
print("thresholds:", {k: round(v, 2) for k, v in TH.items()})

def flag(s):
    reasons = [k for k in TH if s[k] < TH[k]]
    if not s["numbers_ok"]:
        reasons.append("numbers")
    return reasons

print("\nQ2 set (thresholds were chosen here):")
for v in LABELS:
    idx = [i for i in VERDICT if VERDICT[i] == v]
    if idx:
        flagged = sum(bool(flag(dev_sig[i])) for i in idx)
        print(f"  {v:<13} flagged {flagged}/{len(idx)}")

print("\nHeld-out set (not used for thresholds), plain pipeline:")
held_sig = [signals(s) for s, _, _ in HELDOUT]
for i, s in enumerate(held_sig):
    print(f"  [{i}] {'FLAG ' + ','.join(flag(s)) if flag(s) else 'ok':<34} {s['best']}")

thresholds: {'confidence': np.float64(0.68), 'agreement': np.float64(61.61), 'roundtrip': np.float64(45.73)}

Q2 set (thresholds were chosen here):
  ok            flagged 8/15
  minor         flagged 5/5
  fluent-wrong  flagged 6/6

Held-out set (not used for thresholds), plain pipeline:


[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

  [0] FLAG confidence                    Kanaki burned the Madurai.
  [1] FLAG confidence                    Bharathiar was born in Etaipuram.
  [2] FLAG confidence,agreement          She works in Coimbatore.
  [3] FLAG confidence                    Rajaraja Sojan built a great temple in Thanjavur.
  [4] FLAG confidence,roundtrip          Young men wrote some poems.
  [5] FLAG confidence                    Muthul Lakshmi Reddy is a doctor.
  [6] FLAG confidence,roundtrip          He went to Velu.
  [7] FLAG confidence                    Tomorrow Kavya and Praveen will come to Virgo.
  [8] FLAG confidence                    Aruna will come tomorrow.
